#### https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb er brukt som utgangspunkt i denne notebooken.

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [2]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

IMG_RES = (300, 300)

# Cellene under kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

#### The two cells below are based on Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps, Image-Specific Class Saliency Visualisation (https://arxiv.org/abs/1312.6034)

In [3]:
# This method is based on Deep Learning with Python, Third edition. Chapter: 2, Gradient computation,
# and get_gradients from https://keras.io/examples/vision/integrated_gradients/
def get_gradients(img_array, lvl):
    with tf.GradientTape() as tape:
        tape.watch(img_array)
        # Denne veiledningen på reduce_max ble brukt: https://www.tensorflow.org/api_docs/python/tf/math/reduce_max
        score = tf.math.reduce_max(model(img_array)[lvl])
    gradients = tape.gradient(score, img_array)
    return gradients

In [4]:
def get_heatmap(img_array, lvl):
    gradients = get_gradients(img_array, lvl)
    abs_gradients = np.abs(gradients[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    max_gradient = np.max(abs_gradients)
    pixel_gradients = ((abs_gradients / max_gradient) * 255.0)

    values = 0
    heatmap = np.zeros(IMG_RES)

    for i in range(len(pixel_gradients)):
        for j in range(len(pixel_gradients[i])):
            # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
            heatmap[i][j] = np.max(pixel_gradients[i][j])
            values += heatmap[i][j]
    
    return heatmap

### The below cell is strongly based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation

In [5]:
import matplotlib.cm as cm

def superimpose(img_array, heatmap):
    # Uses the "jet" colormap to recolorize the heatmap
    jet = cm.get_cmap("jet")

    jet_colors = jet(np.arange(256))[:, :3]
    jet_colors-=[0,0,0.5]

    # Convertion to int is from: https://www.w3schools.com/python/numpy/numpy_data_types.asp (Converting Data Type on Existing Arrays)
    jet_heatmap = jet_colors[(np.round(heatmap)).astype('i')]

    # Superimposes the heatmap and the original image
    superimposed_img = jet_heatmap + img_array / 3.0
    superimposed_img = keras.utils.array_to_img(superimposed_img)
    return superimposed_img

# Cellene over kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

In [17]:
import imageio

from preprocessing import preprocess_image_tensor

def classify_image(img: Image.Image):

    # Resize to the input image to what MobileNet expects
    # img_resized = img.resize(IMG_RES)
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    arr = arr / 255.0

    # Linjen under er basert på: https://www.tensorflow.org/api_docs/python/tf/keras/ops/convert_to_tensor
    # arr = keras.ops.convert_to_tensor(arr, dtype="float32")

    arr = preprocess_image_tensor(arr, IMG_RES[0])

    # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
    arr = keras.ops.expand_dims(arr, 0)
    # arr = preprocess_input(arr.astype("float32"))

    # Predict!
    preds = model.predict(arr, verbose=0)

    # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
    heatmap = get_heatmap(arr, "lvl1")
    
    # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
    superimposed = superimpose(arr[0], heatmap)

    # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
    superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
    if preds["lvl1"][0][0] > 0.5:
        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap_lvl2 = get_heatmap(arr, "lvl2")
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

    # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
    # imageio.imwrite("Neinieg.png", superimposed)

    car_model_preds = preds["lvl2"][0]

    return [img, superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]



#### Cellen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350

In [18]:
import gradio as gr

def change_visibility(label1, label2):
    try:
        label1["Tesla"]
        # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
        predicted_model = max(label2, key=label2.get)
        return [gr.Image(label="Heatmap som viser hvorfor Tesla"), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True)]
    except:
        return [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla"), gr.Image(visible=False), gr.Label(visible=False)]

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [19]:
# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold") as demo:
    gr.Markdown("## Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold")
    gr.Markdown(
        "Last opp et bilde av et kjøretøy eller velg et eksempel under. "
    )

    with gr.Row():
        components = [0,0,0,0,0]
        components[0] = gr.Image(type="pil", label="Opplastet bilde")
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[1] = gr.Image(type="pil", label="Heatmap", format="png")
        components[2] = gr.Image(type="pil", label="Heatmap", format="png")
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[3] = gr.Label(num_top_classes=1, label="Bilmerke")
            components[4] = gr.Label(num_top_classes=3, label="Bilmodell")

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image, inputs=components[0], outputs=components)
    # Linjen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350
    classify_btn.click(fn=change_visibility, inputs=[components[3], components[4]], outputs=[components[1], components[2], components[4]])

    examples = gr.Examples(
        examples=examples,
        inputs=components[0],
        # outputs=output,
        # fn=classify_image,
        # cache_examples=True
    )

Now we can run it:

In [ ]:
demo.launch(share=True)